# **LaTeX table with the BAO-fit results**

This notebook shows how to create a LaTeX table with the BAO-fit results varying the settings

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["PATH"] = "/global/common/software/nersc9/texlive/2024/bin/x86_64-linux:" + os.environ["PATH"]
os.environ["OMP_NUM_THREADS"] = "1"
import sys
from paths import code_path, save_path
sys.path.append(code_path)
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
from utils_data import GetThetaLimits
from utils_baofit import BAOFitInitializer
from utils_inference import BAOFitChecker, BAOInference
from utils_cosmology import CosmologicalParameters
%matplotlib inline

# helper to select only needed keys for a function
def subset(cfg, *keys):
    return {k: cfg[k] for k in keys}

delta_z = 0.02

recon_list = ["pre", "post"]
tracer_list = ["LRG", "LRG+ELG", "ELG"]
tracer_list = ["LRG", "LRG+ELG"]
# tracer_list = ["LRG"]

def get_bins_removed_list(tracer, delta_z):
    if delta_z != 0.02:
        return []
    if tracer == "LRG":
        nbins = 35
        configs = [
            # ("LRG-full", []),
            ("LRG1", list(range(10, 35))),
            ("LRG2", list(range(0, 10)) + list(range(20, 35))),
            ("LRG3", list(range(0, 20))),
        ]
    elif tracer == "LRG+ELG":
        nbins = 15
        configs = [
            (tracer, []),
        ]
    elif tracer == "ELG":
        nbins = 40
        configs = [
            ("ELG-full", []),
            ("ELG1", list(range(15, 40))),
            ("ELG2", list(range(0, 15))),
        ]
    else:
        return []
    # Individual redshift bins
    for i in range(nbins):
        excluded = [j for j in range(nbins) if j != i]
        configs.append((f"{tracer} bin {i}", excluded))
    return configs

fit_results = {}
likelihoods = {}
likelihoods_nowiggles = {}
significance_detection = {}
z_eff = {}

for recon, tracer in product(recon_list, tracer_list):
    dataset = f"DESIY1_{tracer}_{recon}_deltaz{delta_z}"
    bins_removed_list = get_bins_removed_list(tracer, delta_z)

    if tracer == "ELG":
        theta_width = 3
    else:
        theta_width = 6

    theta_min, theta_max = GetThetaLimits(dataset=dataset, nz_flag="mocks", dynamical_theta_limits=True, theta_width=theta_width, code_path=code_path).get_theta_limits()
    
    for tracer_label, bins_removed in bins_removed_list:

        # 1. Arguments
        base_config = {
            "dataset": dataset,
            "weight_type": 1, # it will not be used...
            "mock_id": "mean", # it will not be used...
            "nz_flag": "mocks",
            "cov_type": "mocks",
            "cosmology_template": "desifid",
            "cosmology_covariance": "desifid",
            "delta_theta": 0.2,
            "theta_min": theta_min,
            "theta_max": theta_max,
            "pow_broadband": [-1, 0],
            "bins_removed": bins_removed,
            "alpha_min": 0.8,
            "alpha_max": 1.2,
            # "alpha_type": "alpha_wigg_only",
            "alpha_type": "alpha_wigg_nowigg",
            "code_path": code_path,
            "save_path": save_path,
        }

        baofit_checker = BAOFitChecker(**base_config) # this will print information related to the detection

        if baofit_checker.is_detection:

            for include_wiggles in ["", "_nowiggles"]:
                config = {
                    **base_config,
                    "include_wiggles": include_wiggles,
                }
    
                # 2. BAO fit initializer. This basically creates the path to load the results
                baofit_initializer = BAOFitInitializer(
                    **subset(config, "include_wiggles", "dataset", "weight_type", "mock_id",
                             "nz_flag", "cov_type", "cosmology_template", "cosmology_covariance",
                             "delta_theta", "theta_min", "theta_max", "pow_broadband",
                             "bins_removed", "alpha_min", "alpha_max", "alpha_type", "save_path"),
                    verbose=False,
                )

                if include_wiggles == "":
                    fit_results[tracer_label, recon] = np.loadtxt(os.path.join(baofit_initializer.path_baofit, "fit_results.txt"))
                    likelihoods[tracer_label, recon] = np.loadtxt(os.path.join(baofit_initializer.path_baofit, "likelihood_data.txt"))
                    significance_detection[tracer_label, recon] = baofit_checker.significance
        
                    z_eff[tracer_label, recon] = BAOInference(baofit_checker, verbose=False).z_eff

                elif include_wiggles == "_nowiggles":
                    likelihoods_nowiggles[tracer_label, recon] = np.loadtxt(os.path.join(baofit_initializer.path_baofit, "likelihood_data.txt"))


In [ ]:
import pandas as pd

rows = []
for (tracer, recon), values in fit_results.items():
    alpha, sigma_alpha, chi2, dof = values

    # format alpha ± sigma
    if sigma_alpha < 100:
        alpha_fmt = f"${alpha:.4f} \\pm {sigma_alpha:.4f}$"
    else:
        alpha_fmt = r"$-$"

    rows.append({
        "tracer": tracer,
        "recon": recon,
        r"$z_{\rm eff}$": f"{z_eff[(tracer, recon)]:.2f}",
        r"$\alpha$": alpha_fmt,
        r"$\chi^2$/dof": f"{chi2:.1f}/{int(dof)}",
        r"significance": f"{significance_detection[(tracer, recon)]:.2f}",
    })

df = pd.DataFrame(rows)

latex_table = df.to_latex(index=False, column_format="l|l|" + "c" * (len(df.columns) - 2))

print(latex_table)


In [ ]:
cosmology_params = CosmologicalParameters("desifid", verbose=False)
cosmo = cosmology_params.get_cosmology()

rows = []

for (tracer, recon), values in fit_results.items():
    alpha, sigma_alpha, chi2, dof = values

    z = z_eff[(tracer, recon)]

    DM_fid = cosmo.comoving_angular_distance(z) / cosmo.h
    rd_fid = cosmo.rs_drag / cosmo.h
    factor = DM_fid / rd_fid

    rows.append({
        "tracer": tracer,
        "recon": recon,
        "z_eff": z,
        "alpha": alpha,
        "alpha_err": sigma_alpha,
        "DM_over_rd": alpha * factor,
        "DM_over_rd_err": sigma_alpha * factor,
        "significance": significance_detection[(tracer, recon)],
    })

df = pd.DataFrame(rows)

colors = {"pre": "royalblue", "post": "crimson"}

main_tracers = ["LRG1", "LRG2", "LRG3"]

markers = {
    "pre": "o",
    "post": "s",
    "main_pre": "D",
    "main_post": "P",
}

x_offset = {"pre": 0.0, "post": 0.0025}

fig, ax = plt.subplots(figsize=(15, 6))

for recon in ["pre", "post"]:
    sub = df[df["recon"] == recon].copy()

    sub["z_plot"] = sub["z_eff"] + x_offset[recon]

    main = sub[sub["tracer"].isin(main_tracers)]
    bins = sub[~sub["tracer"].isin(main_tracers)]

    ax.errorbar(
        bins["z_plot"],
        bins["DM_over_rd"],
        yerr=bins["DM_over_rd_err"],
        fmt=markers[recon],
        linestyle="none",
        color=colors[recon],
        capsize=3,
        alpha=0.6,
        label=f"{recon}"
    )

    ax.errorbar(
        main["z_plot"],
        main["DM_over_rd"],
        yerr=main["DM_over_rd_err"],
        fmt=markers[f"main_{recon}"],
        linestyle="none",
        color=colors[recon],
        capsize=4,
        markersize=9,
        markeredgecolor="black",
        markeredgewidth=1.0,
    )

    if recon == "post":
        for _, row in main.iterrows():
            ax.text(
                row["z_plot"] + 0.003,
                row["DM_over_rd"] + 0.01,
                row["tracer"],
                fontsize=9
            )

z_grid = np.linspace(df["z_eff"].min() - 0.02, df["z_eff"].max() + 0.02, 300)

DM_fid_grid = cosmo.comoving_angular_distance(z_grid) / cosmo.h
rd_fid = cosmo.rs_drag / cosmo.h

fiducial_curve = DM_fid_grid / rd_fid

ax.plot(
    z_grid,
    fiducial_curve,
    color="black",
    linewidth=2,
    label="DESI fiducial"
)

ax.set_xlabel(r"$z$", fontsize=18)
ax.set_ylabel(r"$D_M(z)/r_d$", fontsize=18)

ax.tick_params(axis='both', labelsize=14)

ax.grid(False)

ax.legend(fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

colors = {"LRG1": "#FF9900", "LRG2": "#FF4B00", "LRG3": "#B51D24", "LRG+ELG": "#6959D3"}
i = 0

for (tracer, recon), chi2 in likelihoods.items():
    chi2_nowiggles = likelihoods_nowiggles[tracer, recon]
    if "bin" not in tracer and recon == "post":
        ax.plot(
            chi2[:, 0],
            chi2[:, 1] - chi2[:, 1].min(),
            label=tracer,
            color=colors[tracer],
        )
        ax.plot(
            chi2_nowiggles[:, 0],
            chi2_nowiggles[:, 1] - chi2[:, 1].min(),
            "--",
            color=colors[tracer],
        )
        i += 1

ax.set_xlabel(r"$\alpha$", fontsize=18)
ax.set_ylabel(r"$\Delta\chi^2$", fontsize=18)
ax.tick_params(axis='both', labelsize=14)

ax.set_xlim(0.8, 1.2)
ax.set_ylim(bottom=0)

ax.legend()
plt.tight_layout()
